Data import

In [1]:
import pandas as pd 

In [2]:
Sales           = pd.read_parquet('PraquetDATA/Sales.parquet')
Purchase        = pd.read_parquet('PraquetDATA/Purchase.parquet')
Future_Price    = pd.read_csv('CSVDATA/2017PurchasePricesDec.csv')
Purchase_invoice= pd.read_csv('CSVDATA/InvoicePurchases12312016.csv')
Invoice_Begin   = pd.read_csv('CSVDATA/BegInvFINAL12312016.csv')
Invoice_End     = pd.read_csv('CSVDATA/EndInvFINAL12312016.csv')

Fill some Nan Values

In [102]:
Sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 12825363 entries, 0 to 12825362
Data columns (total 15 columns):
 #   Column          Dtype         
---  ------          -----         
 0   InventoryId     category      
 1   Store           int8          
 2   Brand           int32         
 3   Description     category      
 4   Size            category      
 5   SalesQuantity   int16         
 6   SalesDollars    float16       
 7   SalesPrice      float16       
 8   SalesDate       datetime64[us]
 9   Volume          int16         
 10  Classification  int8          
 11  ExciseTax       float16       
 12  VendorNo        int32         
 13  VendorName      category      
 14  Week            uint8         
dtypes: category(4), datetime64[us](1), float16(3), int16(2), int32(2), int8(2), uint8(1)
memory usage: 471.5 MB


In [3]:
Purchase.loc[Purchase['Brand']==15365 , 'Size'] = '750mL'
Purchase.loc[Purchase['Brand']==3121 , 'Size'] = '750mL'
Purchase.loc[Purchase['Brand']==5678 , 'Size'] = '750mL'

Changing Date Dtypes

In [4]:
Invoice_Begin['startDate']  = pd.to_datetime(Invoice_Begin['startDate']).dt.normalize()
Invoice_End['endDate']      = pd.to_datetime(Invoice_End['endDate']).dt.normalize()
Purchase_invoice[['InvoiceDate','PODate','PayDate']]=Purchase_invoice[['InvoiceDate','PODate','PayDate']].apply(pd.to_datetime).apply(lambda x: x.dt.normalize())

Reconciliation of Usual SKU

In [5]:
Permanent_SKU= set(set(Sales['Brand']) & set(Invoice_Begin['Brand']) & set(Invoice_End['Brand']))
print(len(Permanent_SKU))

6933


In [6]:
Sales[Sales['Brand'].isin(Permanent_SKU)][["SalesDollars","SalesQuantity"]].astype('Float64').sum()

SalesDollars     430903999.739258
SalesQuantity          30806116.0
dtype: Float64

Thank u chat gpt (this is a reference for slicing checks)

In [7]:
sales_ids = set(Sales['Brand'].unique())
begin_ids = set(Invoice_Begin['Brand'].unique())
end_ids   = set(Invoice_End['Brand'].unique())

print("Sales:", len(sales_ids))
print("Begin:", len(begin_ids))
print("End:", len(end_ids))

print("All 3:", len(sales_ids & begin_ids & end_ids))
print("Begin only:", len(begin_ids - end_ids))
print("End only:", len(end_ids - begin_ids))

Sales: 11237
Begin: 8094
End: 9653
All 3: 6933
Begin only: 1106
End only: 2665


items that got removed mid year

In [8]:
#DS discontinued stock
DS = set(Invoice_Begin['Brand']) - set(Invoice_End['Brand'])
len(DS)

1106

In [9]:
All_Sold_DS =  set(Sales[Sales['Brand'].isin(DS)]['Brand'])
print(len(All_Sold_DS))

1090


In [10]:
All_Purchased_DS = DS & set(Purchase['Brand'])
print(len(All_Purchased_DS))

631


In [11]:
All_Purchased_Sold_DS = All_Sold_DS & All_Purchased_DS
print(len(All_Purchased_Sold_DS))

631


In [12]:
All_Non_Purchased_DS = All_Sold_DS - All_Purchased_DS
print(len(All_Non_Purchased_DS))

459


In [13]:
All_Vanished_DS = DS - All_Sold_DS
print(len(All_Vanished_DS))

16


In [14]:
#overview of removed mid year
print('all Sold value & QTE')
print('Total SKUs: ' ,len(All_Sold_DS),' S/Q ',Sales[Sales['Brand'].isin(All_Sold_DS)]['SalesDollars'].astype('int64').sum(),' & ' , Sales[Sales['Brand'].isin(All_Sold_DS)]['SalesQuantity'].astype('int64').sum()  )
print('all Sold/Purchased value & QTE')
print('Total SKUs: ' ,len(All_Purchased_Sold_DS),' S/Q ',Sales[Sales['Brand'].isin(All_Purchased_Sold_DS)]['SalesDollars'].astype('int64').sum(),' & ' , Sales[Sales['Brand'].isin(All_Purchased_Sold_DS)]['SalesQuantity'].astype('int64').sum()  )
print('all Non Purchased value & QTE')
print('Total SKUs: ' ,len(All_Non_Purchased_DS),' S/Q ',Sales[Sales['Brand'].isin(All_Non_Purchased_DS)]['SalesDollars'].astype('int64').sum(),' & ' , Sales[Sales['Brand'].isin(All_Non_Purchased_DS)]['SalesQuantity'].astype('int64').sum()  )
print('all Purchased value & QTE')
print('Total SKUs: ' ,len(All_Purchased_DS),' S/Q ',Purchase[Purchase['Brand'].isin(All_Purchased_DS)]['Dollars'].astype('int64').sum(),' & ' , Purchase[Purchase['Brand'].isin(All_Purchased_DS)]['Quantity'].astype('int64').sum()  )


all Sold value & QTE
Total SKUs:  1090  S/Q  3023560  &  221787
all Sold/Purchased value & QTE
Total SKUs:  631  S/Q  2885225  &  216113
all Non Purchased value & QTE
Total SKUs:  459  S/Q  138335  &  5674
all Purchased value & QTE
Total SKUs:  631  S/Q  1181983  &  115681


In [15]:
#starting inventory for DS
print('all DS  I QTE')
print('Total SKUs: ' ,len(DS),' Q ', Invoice_Begin[Invoice_Begin['Brand'].isin(DS)]['onHand'].sum())
print('all Sold/Purchased I QTE')
print('Total SKUs: ' ,len(All_Purchased_Sold_DS),' Q ', Invoice_Begin[Invoice_Begin['Brand'].isin(All_Purchased_Sold_DS)]['onHand'].sum()  )
print('all Non Purchased I QTE')
print('Total SKUs: ' ,len(All_Non_Purchased_DS),' Q ', Invoice_Begin[Invoice_Begin['Brand'].isin(All_Non_Purchased_DS)]['onHand'].sum()  )
print('all Purchased  I QTE')
print('Total SKUs: ' ,len(All_Purchased_DS),' Q ', Invoice_Begin[Invoice_Begin['Brand'].isin(All_Purchased_DS)]['onHand'].sum()  )
print('all  Vanished I QTE')
print('Total SKUs: ' ,len(All_Vanished_DS),' Q ', Invoice_Begin[Invoice_Begin['Brand'].isin(All_Vanished_DS)]['onHand'].sum()  )

all DS  I QTE
Total SKUs:  1106  Q  106106
all Sold/Purchased I QTE
Total SKUs:  631  Q  100432
all Non Purchased I QTE
Total SKUs:  459  Q  5674
all Purchased  I QTE
Total SKUs:  631  Q  100432
all  Vanished I QTE
Total SKUs:  16  Q  0


items that got Added mid year

In [16]:
#AAS all added stock
AAS = set(set(Invoice_End['Brand'])-set(Invoice_Begin['Brand']))
print(Invoice_End[Invoice_End['Brand'].isin(AAS)]['Brand'].nunique())

2665


In [17]:
AAS_Sold = set(Sales[Sales['Brand'].isin(AAS)]['Brand'])
print('Sold:   ' , len(AAS_Sold))

Sold:    2470


In [18]:
AAS_Purchased = set(Purchase[Purchase['Brand'].isin(AAS)]['Brand'])
print('Purchased: ' ,len(AAS_Purchased))

Purchased:  2647


In [19]:
AAS_spawned_end=set (Invoice_End[Invoice_End['Brand'].isin(AAS_Purchased-AAS_Sold)])
print('Purchased: ' ,len(AAS_spawned_end))

Purchased:  9


In [20]:
AAS_Sold_Purchased = set(AAS_Purchased&AAS_Sold)
print('Purchased: ' ,len(AAS_Sold_Purchased))

Purchased:  2470


In [21]:
AAS_spawned=set (AAS_Purchased-AAS_Sold)
print('Purchased: ' ,len(AAS_spawned))

Purchased:  177


In [22]:
AAS_Not_Purchased = AAS - AAS_Purchased
print('Purchased: ' ,len(AAS_Not_Purchased))

Purchased:  18


In [23]:
#overview of removed mid year
print('all Sold value & QTE')
print('Total SKUs: ' ,len(AAS_Sold),' S/Q ',Sales[Sales['Brand'].isin(AAS_Sold)]['SalesDollars'].astype('int64').sum(),' & ' , Sales[Sales['Brand'].isin(AAS_Sold)]['SalesQuantity'].astype('int64').sum()  )
print('all Sold/Purchased value & QTE')
print('Total SKUs: ' ,len(AAS_Sold_Purchased),' S/Q ',Sales[Sales['Brand'].isin(AAS_Sold_Purchased)]['SalesDollars'].astype('int64').sum(),' & ' , Sales[Sales['Brand'].isin(AAS_Sold_Purchased)]['SalesQuantity'].astype('int64').sum()  )
print('all Non Purchased value & QTE')
print('Total SKUs: ' ,len(AAS_Not_Purchased),' S/Q ',Sales[Sales['Brand'].isin(AAS_Not_Purchased)]['SalesDollars'].astype('int64').sum(),' & ' , Sales[Sales['Brand'].isin(AAS_Not_Purchased)]['SalesQuantity'].astype('int64').sum()  )
print('all Purchased value & QTE')
print('Total SKUs: ' ,len(AAS_Purchased),' S/Q ',Purchase[Purchase['Brand'].isin(AAS_Purchased)]['Dollars'].astype('int64').sum(),' & ' , Purchase[Purchase['Brand'].isin(AAS_Purchased)]['Quantity'].astype('int64').sum()  )


all Sold value & QTE
Total SKUs:  2470  S/Q  15931441  &  1769180
all Sold/Purchased value & QTE
Total SKUs:  2470  S/Q  15931441  &  1769180
all Non Purchased value & QTE
Total SKUs:  18  S/Q  0  &  0
all Purchased value & QTE
Total SKUs:  2647  S/Q  18261203  &  2321385


In [24]:
#starting inventory for DS
print('all DS  I QTE')
print('Total SKUs: ' ,len(AAS),' Q ', Invoice_End[Invoice_End['Brand'].isin(AAS)]['onHand'].sum())
print('all Sold/Purchased I QTE')
print('Total SKUs: ' ,len(AAS_Sold_Purchased),' Q ', Invoice_End[Invoice_End['Brand'].isin(AAS_Sold_Purchased)]['onHand'].sum()  )
print('all Non Purchased I QTE')
print('Total SKUs: ' ,len(AAS_Not_Purchased),' Q ', Invoice_End[Invoice_End['Brand'].isin(AAS_Not_Purchased)]['onHand'].sum()  )
print('all Purchased  I QTE')
print('Total SKUs: ' ,len(AAS_Purchased),' Q ', Invoice_End[Invoice_End['Brand'].isin(AAS_Purchased)]['onHand'].sum()  )
print('all  Vanished I QTE')
print('Total SKUs: ' ,len(AAS_spawned),' Q ', Invoice_End[Invoice_End['Brand'].isin(AAS_spawned)]['onHand'].sum()  )

all DS  I QTE
Total SKUs:  2665  Q  552205
all Sold/Purchased I QTE
Total SKUs:  2470  Q  539501
all Non Purchased I QTE
Total SKUs:  18  Q  0
all Purchased  I QTE
Total SKUs:  2647  Q  552205
all  Vanished I QTE
Total SKUs:  177  Q  12704


Items that entered and left same year

In [25]:
test_Sku= set(Sales['Brand'])-set(Invoice_Begin['Brand'])-set(Invoice_End['Brand'])
print(len(test_Sku))

744


Stock reconciliation

In [26]:
Store_recon =  Sales.groupby(['Store','Brand'], observed=True)['SalesQuantity'].sum().reset_index().merge(
    Purchase.groupby(['Store','Brand'], observed=True)['Quantity'].sum().reset_index(),on=['Store','Brand'],how='outer').merge(
        Invoice_Begin.groupby(['Store','Brand'], observed=True)['onHand'].sum().reset_index(),on=['Store','Brand'],how='outer'
    ).merge(Invoice_End.groupby(['Store','Brand'], observed=True)['onHand'].sum().reset_index(),on=['Store','Brand'],how='outer'
    )

In [27]:
Store_recon = Store_recon [['Store','Brand','onHand_x','Quantity','SalesQuantity','onHand_y']]

In [28]:
Store_recon[['onHand_x','Quantity','SalesQuantity','onHand_y']].sum()

onHand_x          4219275.0
Quantity         33584377.0
SalesQuantity    32917876.0
onHand_y          4885776.0
dtype: float64

In [29]:
Store_recon = Store_recon.fillna(0)

In [30]:
Store_recon['C_end']= Store_recon['onHand_x'] + Store_recon['Quantity'] - Store_recon['SalesQuantity']

In [31]:
Store_recon['DIFF']= Store_recon['C_end']-Store_recon['onHand_y']

In [32]:
Store_recon[Store_recon['DIFF'] != 0] 

,Store,Brand,onHand_x,Quantity,SalesQuantity,onHand_y,C_end,DIFF


In [33]:
brand_recon = Store_recon.groupby('Brand')[['onHand_x',"Quantity" ,'SalesQuantity','onHand_y','C_end','DIFF']].sum().reset_index()

In [34]:
brand_recon[brand_recon['Brand'].isin(AAS)].sum()

Brand            52888904.0
onHand_x                0.0
Quantity          2321385.0
SalesQuantity     1769180.0
onHand_y           552205.0
C_end              552205.0
DIFF                    0.0
dtype: float64

In [35]:
brand_recon[brand_recon['Brand'].isin(Permanent_SKU)].sum()

Brand            121930620.0
onHand_x           4113014.0
Quantity          31026506.0
SalesQuantity     30806116.0
onHand_y           4333404.0
C_end              4333404.0
DIFF                     0.0
dtype: float64

In [36]:
brand_recon[brand_recon['Brand'].isin(DS)].sum()

Brand            17507488.0
onHand_x           106106.0
Quantity           115681.0
SalesQuantity      221787.0
onHand_y                0.0
C_end                   0.0
DIFF                    0.0
dtype: float64

In [37]:
brand_recon[brand_recon['Brand'].isin(test_Sku)].sum()

Brand            14488151.0
onHand_x                0.0
Quantity           120793.0
SalesQuantity      120793.0
onHand_y                0.0
C_end                   0.0
DIFF                    0.0
dtype: float64

Value recon for later use 

In [38]:
Invoice_Begin['I_Value']    = Invoice_Begin['onHand']   * Invoice_Begin['Price']   
Invoice_End['S_Value']      =Invoice_End['onHand']   * Invoice_End['Price']   

In [39]:
Pricesx = Invoice_Begin[['Brand','Price']].drop_duplicates()

In [40]:
Pricesy = Purchase[['Brand','PurchasePrice']].drop_duplicates()

In [41]:
Price = set(Pricesx['Brand']) | set(Pricesy['Brand'])

In [42]:
Price=pd.DataFrame(
    {'Brand':list(Price)}
)

In [43]:
Price = Price.merge(Pricesx, on= 'Brand' , how = 'left').merge(Pricesy, on= 'Brand' , how = 'left')

ABC analysis

Store Competitiveness

In [45]:
S_Competitive= Sales.groupby('Store').sum(numeric_only=True).sort_values(by=['SalesDollars','SalesQuantity'],ascending=[False,False])[['SalesQuantity','SalesDollars']]

In [46]:
S_Competitive['percentage']       = (S_Competitive['SalesDollars']/S_Competitive['SalesDollars'].sum())*100
S_Competitive['percentage_Cumul'] = S_Competitive['percentage'].cumsum()
S_Competitive['ABC Sales']        = S_Competitive['percentage_Cumul'].apply(lambda x :'A' if x <= 80 else ('B' if x <=95 else 'C') )

In [47]:
S_Competitive.reset_index(inplace=True)

In [48]:
S_Competitive

,Store,SalesQuantity,SalesDollars,percentage,percentage_Cumul,ABC Sales
0,76,1573145,2.545211e+07,5.630141,5.630141,A
1,73,1365482,2.173654e+07,4.808237,10.438377,A
2,34,1365123,2.092919e+07,4.629647,15.068024,A
3,38,1320833,1.945203e+07,4.302891,19.370914,A
4,66,1118778,1.782609e+07,3.943225,23.314140,A
...,...,...,...,...,...,...
75,37,84773,1.058592e+06,0.234166,99.414062,C
76,81,73129,9.349641e+05,0.206819,99.620880,C
77,29,68434,7.279115e+05,0.161018,99.781898,C
78,26,45839,5.663711e+05,0.125284,99.907181,C


Products Competitiveness (company overall)

Permanent Sku

In [71]:
len(Permanent_SKU)

6933

In [54]:
Company_ABC = Sales[Sales['Brand'].isin(Permanent_SKU)].groupby('Brand').sum(numeric_only=True).sort_values(by=['SalesDollars','SalesQuantity'],ascending=[False,False])[['SalesQuantity','SalesDollars']]

In [55]:
Company_ABC.reset_index(inplace=True)
Company_ABC

,Brand,SalesQuantity,SalesDollars
0,1233,142049,5.102831e+06
1,3405,160247,4.819520e+06
2,8068,187140,4.537913e+06
3,4261,200412,4.475443e+06
4,3545,135838,4.223518e+06
...,...,...,...
6928,21850,7,1.253027e+01
6929,19138,1,9.992188e+00
6930,3678,4,9.960938e+00
6931,3053,8,7.921875e+00


In [56]:
Company_ABC['percentage']       = (Company_ABC['SalesDollars']/Company_ABC['SalesDollars'].sum())*100
Company_ABC['percentage_Cumul'] = Company_ABC['percentage'].cumsum()
Company_ABC['ABC']        = Company_ABC['percentage_Cumul'].apply(lambda x :'A' if x <= 50 else ('B' if x <=80 else 'C') )

In [57]:
Company_ABC['ABC'].value_counts()

ABC
C    5596
B    1004
A     333
Name: count, dtype: int64

Xyz

In [91]:
Sales['Week']       = (((Sales['SalesDate'].dt.dayofyear - 1) // 7) + 1).clip(upper=52).astype('uint8')
Purchase['Week']    = (((Purchase['ReceivingDate'].dt.dayofyear - 1) // 7) + 1).clip(upper=52).astype('uint8')

In [59]:
weekly_sales = (
    Sales[[ 'Brand', 'Week', 'SalesQuantity']]
    .groupby([ 'Brand', 'Week'])['SalesQuantity']
    .sum()  
)
weekly_grid = weekly_sales.unstack(level='Week', fill_value=0)

In [60]:
weekly_grid['Mean'] =weekly_grid.mean(axis=1)
weekly_grid['Std']  =weekly_grid.std(axis=1)
weekly_grid['CV']   =(weekly_grid['Std']/weekly_grid['Mean'])*100

In [83]:
weekly_grid.loc[weekly_grid['CV'] <= 50, 'XYZ']   = 'X'
weekly_grid.loc[weekly_grid['CV'] > 50, 'XYZ']    = 'Y'
weekly_grid.loc[weekly_grid['CV'] > 100, 'XYZ']   = 'Z'

In [64]:
weekly_grid.reset_index(inplace=True)

In [84]:
ABC_XYZ = Company_ABC.merge(weekly_grid[['Brand','XYZ']],on=['Brand'],how='left')

In [85]:
ABC_XYZ['Tier']=ABC_XYZ['ABC']+ABC_XYZ['XYZ']

In [89]:
ABC_XYZ.to_excel('hat.xlsx')

Sales overview

In [99]:
Sales.groupby('Week')[['SalesDollars','SalesQuantity']].sum().to_excel('SalesWeel.xlsx')


In [100]:
Purchase.groupby('Week')[['Dollars','Quantity']].sum().to_excel('PurchaseWeel.xlsx')

Making company overall based on product classification

In [ ]:
Company_ABC_one = Company_ABC[Company_ABC['Classification']== 1]
Company_ABC_one['percentage']       = (Company_ABC_one['SalesDollars']/Company_ABC_one['SalesDollars'].sum())*100
Company_ABC_one['percentage_Cumul'] = Company_ABC_one['percentage'].cumsum()
Company_ABC_one['ABC']        = Company_ABC_one['percentage_Cumul'].apply(lambda x :'A' if x <= 50 else ('B' if x <=80 else 'C') )

In [ ]:
Company_ABC_Two = Company_ABC[Company_ABC['Classification']== 2]
Company_ABC_Two['percentage']       = (Company_ABC_Two['SalesDollars']/Company_ABC_Two['SalesDollars'].sum())*100
Company_ABC_Two['percentage_Cumul'] = Company_ABC_Two['percentage'].cumsum()
Company_ABC_Two['ABC Sales']        = Company_ABC_Two['percentage_Cumul'].apply(lambda x :'A' if x <= 80 else ('B' if x <=95 else 'C') )


Stores Best selling products (ABC)

In [ ]:
Sales.head(5)

In [ ]:
Store_ABC = pd.DataFrame([])
Store_ABC = Sales.groupby(by=['Store','Brand']).sum(numeric_only=True)[['SalesQuantity','SalesDollars']].reset_index()

In [ ]:
Store_ABC.sort_values(by=['Store','SalesDollars','SalesQuantity'],ascending=[True,False,False],inplace=True)
Store_ABC.reset_index(drop=True,inplace=True)

In [ ]:
Store_ABC['SumSalesByStore'] = Store_ABC.groupby(by='Store')['SalesDollars'].transform('sum')

In [ ]:
Store_ABC['percentage'] = (Store_ABC['SalesDollars']/Store_ABC['SumSalesByStore'])*100

In [ ]:
Store_ABC['percentage_Cumul'] = Store_ABC.groupby(by='Store')['percentage'].cumsum()

In [ ]:
Store_ABC['ABC'] = Store_ABC['percentage_Cumul'].apply(lambda x :'A' if x <= 50 else ('B' if x <=80 else 'C'))

In [ ]:
Store_ABC

Store XYZ

In [ ]:
Sales['Week'] = (((Sales['SalesDate'].dt.dayofyear - 1) // 7) + 1).clip(upper=52).astype('uint8')

In [ ]:
weekly_sales = (
    Sales[['Store', 'Brand', 'Week', 'SalesQuantity']]
    .groupby(['Store', 'Brand', 'Week'])['SalesQuantity']
    .sum()  
)
weekly_grid = weekly_sales.unstack(level='Week', fill_value=0)

In [ ]:
weekly_grid['Mean'] =weekly_grid.mean(axis=1)
weekly_grid['Std']  =weekly_grid.std(axis=1)
weekly_grid['CV']   =(weekly_grid['Std']/weekly_grid['Mean'])*100

In [ ]:
weekly_grid.loc[weekly_grid['CV'] <= 50, 'XYZ']   = 'X'
weekly_grid.loc[weekly_grid['CV'] > 50, 'XYZ']    = 'Y'
weekly_grid.loc[weekly_grid['CV'] > 100, 'XYZ']   = 'Z'

In [ ]:
weekly=weekly_grid.reset_index()

Fusion card (ABC_XYZ)

In [ ]:
STORE_ABC_XYZ = Store_ABC.merge(weekly[['Store','Brand','XYZ']],on=['Store','Brand'],how='left')

In [ ]:
STORE_ABC_XYZ['Tier']=STORE_ABC_XYZ['ABC']+STORE_ABC_XYZ['XYZ']

In [ ]:
STORE_ABC_XYZ[STORE_ABC_XYZ['Store']==1]['Tier'].value_counts()

In [ ]:
STORE_ABC_XYZ =  STORE_ABC_XYZ.merge(Sales[['Brand','Size','Description']].drop_duplicates(),on='Brand',how='left')

In [ ]:
STORE_ABC_XYZ['ID'] = (STORE_ABC_XYZ['Description'].astype(str)+'(' + STORE_ABC_XYZ['Size'].astype(str) +')').astype('category')

FUSION DSAHBOARD

In [ ]:
STORE_ABC_XYZ.to_csv('Store.csv')

Demand Forcasting

Stock checking 

In [ ]:
intial_stock=Invoice_Begin.groupby(['Store','Brand']).sum(numeric_only=True)[['onHand']].reset_index()
Plus = Purchase.groupby(['Store','Brand']).sum(numeric_only=True)[['Quantity']].reset_index()
Minus=Sales.groupby(['Store','Brand']).sum(numeric_only=True)[['SalesQuantity']].reset_index()
Stock_variation= Minus.merge(Plus , on=['Store','Brand'],how='outer').merge(intial_stock , on=['Store','Brand'],how='outer')
Stock_variation[['SalesQuantity','Quantity','onHand']] = Stock_variation[['SalesQuantity','Quantity','onHand']].fillna(0)

In [ ]:
Stock_variation = Stock_variation[['Store','Brand','onHand','Quantity','SalesQuantity']]

In [ ]:
Stock_variation['CFS'] = Stock_variation['onHand']+Stock_variation['Quantity']-Stock_variation['SalesQuantity'] #calculated final stock
Endstock =Invoice_End.groupby(['Store','Brand']).sum(numeric_only=True)[['onHand']].reset_index()
Stock_variation = Stock_variation.merge(Endstock , on=['Store','Brand'],how='outer')
Stock_variation[['SalesQuantity','Quantity','onHand_x','CFS','onHand_y']] = Stock_variation[['SalesQuantity','Quantity','onHand_x','CFS','onHand_y']].fillna(0)
Stock_variation = Stock_variation.rename(
    columns={
        "onHand_x"      : "Beginning_Inventory",
        "Quantity"      : "Purchased_Qty",
        "SalesQuantity" : "Sales_Qty",
        "CFS"           : "Calculated_Ending",
        'onHand_y'      :'Ending_Quantity'
    }
)
Stock_variation['Shrinkage']= Stock_variation['Calculated_Ending']-Stock_variation['Ending_Quantity']



Lead table

In [ ]:
Sales['Month']=(Sales['SalesDate'].dt.month).astype('int8')

In [ ]:
Purchase['lead']=(Purchase['ReceivingDate']-Purchase['PODate']).dt.days.astype('int8')

In [ ]:
Lead_table = Purchase.groupby(['Brand','VendorName'])['lead'].mean().reset_index()

In [ ]:
Lead_table = Lead_table.merge(Purchase.groupby(['Brand','VendorName'])['lead'].std().rename('STD').reset_index(),on=['Brand','VendorName'],how='left').fillna(0)

In [ ]:
Lead_table['CV'] = Lead_table['STD'] / Lead_table['lead']

In [ ]:
Lead_table.loc[Lead_table['CV'] <=0.2 , 'XYZ']   = 'X'
Lead_table.loc[Lead_table['CV'] > 0.2 , 'XYZ']   = 'Y'
Lead_table.loc[Lead_table['CV'] > 1   , 'XYZ']   = 'Z'
Lead_table['XYZ'] = Lead_table['XYZ'].astype('category')

Demand forcasting

In [ ]:
Company_ABC

In [ ]:
weekly_sales = (
    Sales[['Brand', 'Week', 'SalesQuantity']]
    .groupby(['Brand', 'Week'])['SalesQuantity']
    .sum()  
)
weekly_grid = weekly_sales.unstack(level='Week', fill_value=0)
weekly_grid['Mean'] =weekly_grid.mean(axis=1)
weekly_grid['Std']  =weekly_grid.std(axis=1)
weekly_grid['CV']   =(weekly_grid['Std']/weekly_grid['Mean'])*100
weekly_grid.loc[weekly_grid['CV'] <= 50, 'XYZ']   = 'X'
weekly_grid.loc[weekly_grid['CV'] > 50, 'XYZ']    = 'Y'
weekly_grid.loc[weekly_grid['CV'] > 100, 'XYZ']   = 'Z'


In [ ]:
weekly=weekly_grid.reset_index()

In [ ]:
weekly

In [ ]:
C_overview= Company_ABC.merge(weekly[['Brand','Mean','Std','XYZ']],on='Brand',how='left')

In [ ]:
C_overview['Tier']=C_overview['ABC']+C_overview['XYZ']

In [ ]:
C_overview

In [ ]:
Invoice_Begin['Brand'].nunique()

In [ ]:
Invoice_End['Brand'].nunique()

In [ ]:
Sales['Brand'].nunique()